In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
int_inns = [some_inns]

dt_inns = []

for inn in int_inns:
    dt_inns.append(str(inn))

In [ ]:
dt_to_process = pd.read_excel("/home/datalab/nfs/deepfm/data/test_22_07_2025/Покупатели для расчета рекомендаций.xlsx")
dt_inns = dt_to_process["nn_DT"].unique()
dt_inns

In [4]:
import pyarrow.parquet as pq
import pandas as pd
import gzip
import pickle

kt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_kt/data"
labels_df = pq.read_table(kt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_kt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

dt_indexer_path = "/home/datalab/nfs/deepfm/data/train_21/arnsdpsbx_t_team_fin_adviser.indexer_dt/data"
labels_df = pq.read_table(dt_indexer_path).to_pandas()
labels_array = labels_df["labelsArray"][0]
inn_dt_to_index = {
    labels_array[0][i]: i for i in range(len(labels_array[0]))
}

In [5]:
dt_inns_to_index = {}
count = 0

for inn in dt_inns:
    if not str(inn) in inn_dt_to_index:
        # print(inn)
        count += 1

print(count)

5


In [6]:
import sys
import os

sys.path.append("/home/datalab/nfs/deepfm/")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pandas as pd
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import json
from tqdm import tqdm

from src.datasets import StreamDataset
from src.datasets.collate import collate_fn
from src.models import DeepFM

In [7]:
kt_features = [    
    "inn_kt_index",
    "okved_cd_kt_index",
    "okato_cd_kt_index",
    "bic_kt_34_index",
    "bic_kt_56_index",
    "bic_kt_79_index",
    "num_kt_13_index",
    "num_kt_45_index",
    "num_kt_68_index",
    "okved_cd_kt_lvl1_index",
    "okved_cd_kt_lvl2_index",
    "okved_cd_kt_lvl3_index"]

kt_double_features = [
    "kt_avg_sum",
    "kt_stddev_sum",
    "kt_min_sum",
    "kt_max_sum",
    "kt_median_sum",
    "kt_skewness_sum",
    "kt_buyers_count"]

dt_features =  [    
    "inn_dt_index",
    "okved_cd_dt_index",
    "okato_cd_dt_index",
    "bic_dt_34_index",
    "bic_dt_56_index",
    "bic_dt_79_index",
    "num_dt_13_index",
    "num_dt_45_index",
    "num_dt_68_index",
    "okved_cd_dt_lvl1_index",
    "okved_cd_dt_lvl2_index",
    "okved_cd_dt_lvl3_index"]

dt_double_features = [
    "dt_avg_sum",
    "dt_stddev_sum",
    "dt_min_sum",
    "dt_max_sum",
    "dt_median_sum",
    "dt_skewness_sum",
    "dt_buyers_count"]

label_column = "label"

In [8]:
device = torch.device("cuda")

with open("/home/datalab/nfs/deepfm/data/train_21/user_feature_sizes.json", "r") as f:
    user_feature_sizes = json.load(f)

with open("/home/datalab/nfs/deepfm/data/train_21/item_feature_sizes.json", "r") as f:
    item_feature_sizes = json.load(f)

model = DeepFM(
    embed_dim=128,
    num_user_double_feats=7,
    num_item_double_feats=7,
    user_feature_sizes=user_feature_sizes,
    item_feature_sizes=item_feature_sizes,
).to(device)

checkpoint = torch.load("/home/datalab/nfs/deepfm/deepfm_logs/train_21_boevoy_zapusk_vse_fichi/model_best.pth", device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

DeepFM(
  (user_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_dt_index): Embedding(2798745, 128)
      (okved_cd_dt_index): Embedding(2643, 51)
      (okato_cd_dt_index): Embedding(60671, 128)
      (bic_dt_34_index): Embedding(84, 9)
      (bic_dt_56_index): Embedding(56, 7)
      (bic_dt_79_index): Embedding(96, 9)
      (num_dt_13_index): Embedding(14, 3)
      (num_dt_45_index): Embedding(26, 5)
      (num_dt_68_index): Embedding(4, 2)
      (okved_cd_dt_lvl1_index): Embedding(91, 9)
      (okved_cd_dt_lvl2_index): Embedding(101, 10)
      (okved_cd_dt_lvl3_index): Embedding(69, 8)
    )
  )
  (item_embed): FeatureEmbedding(
    (embeddings): ModuleDict(
      (inn_kt_index): Embedding(2364265, 128)
      (okved_cd_kt_index): Embedding(2580, 50)
      (okato_cd_kt_index): Embedding(59481, 128)
      (bic_kt_34_index): Embedding(79, 8)
      (bic_kt_56_index): Embedding(51, 7)
      (bic_kt_79_index): Embedding(96, 9)
      (num_kt_13_index): Embedding(17, 4)
  

In [9]:
import gzip, pickle

with gzip.open("../data/train_21/dt_embeddings_dict.pkl.gz", "rb") as f: dt_embs = pickle.load(f)
with gzip.open("../data/train_21/kt_embeddings_dict.pkl.gz", "rb") as f: kt_embs = pickle.load(f)
with gzip.open("../data/train_21/dt_features_dict.pkl.gz", "rb") as f: dt_feat = pickle.load(f)
with gzip.open("../data/train_21/kt_features_dict.pkl.gz", "rb") as f: kt_feat = pickle.load(f)

In [10]:
dt_inns_for_test = []

dt_inns_for_test = dt_inns

In [11]:
dt_embeddings = {}
dt_attentions = {}
cold_recs = set()

for i, inn in tqdm(enumerate(dt_inns_for_test)):
    if inn in inn_dt_to_index:
        uid = inn_dt_to_index.get(inn, -1)
        data = dt_feat.get(uid, {})
        cat = torch.tensor([[data.get(f, 0) for f in dt_features]], dtype=torch.long, device="cuda")
        cont = torch.tensor([[data.get(f, 0.0) for f in dt_double_features]], dtype=torch.float32, device="cuda")
        emb0 = torch.tensor(dt_embs.get(uid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)
        with torch.no_grad():
            emb, attention = model.embed_user(cat, cont, emb0)
        dt_embeddings[uid] = emb.squeeze(0).cpu().numpy()
        dt_attentions[uid] = attention.squeeze(0).cpu().numpy()
    else:
        # print(inn)
        cold_recs.add(inn)
        uid = inn_dt_to_index.get(inn, -1)
        data = dt_feat.get(uid, {})
        cat = torch.tensor([[data.get(f, 0) for f in dt_features]], dtype=torch.long, device="cuda")
        cont = torch.tensor([[data.get(f, 0.0) for f in dt_double_features]], dtype=torch.float32, device="cuda")
        emb0 = torch.tensor(dt_embs.get(uid, np.zeros(256)), dtype=torch.float32, device="cuda").unsqueeze(0)
        with torch.no_grad():
            emb, attention = model.embed_user(cat, cont, emb0)
        dt_embeddings[inn] = emb.squeeze(0).cpu().numpy()
        dt_attentions[inn] = attention.squeeze(0).cpu().numpy()

68it [00:00, 202.74it/s]


In [27]:
import gzip
import pickle

def save_compressed(path, embeddings):
    with gzip.open(path, "wb") as f:
        pickle.dump(embeddings, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("saved")

dt_embeds_path = "../data/train_21/embeddings/07_08_2025_dt_embeddings_test.pkl.gz"
# kt_embeds_path = "../data/train_21/embeddings/kt_embeddings_test.pkl.gz"

# dt_attns_path = "../data/train_21/attentions/dt_attentions_test.pkl.gz"
# kt_attns_path = "../data/train_21/attentions/kt_attentions_test.pkl.gz"

# сохраняем наши словарики
save_compressed(dt_embeds_path, dt_embeddings)
# save_compressed(kt_embeds_path, kt_embeddings)
# save_compressed(dt_attns_path, dt_attentions)
# save_compressed(kt_attns_path, kt_attentions)

saved


### Making recommendations table (after finding nearest neighbors)

In [68]:
index_to_inn_kt = {index: inn for inn, index in inn_kt_to_index.items()}
index_to_inn_dt = {index: inn for inn, index in inn_dt_to_index.items()}

In [69]:
recommendations_path = "/home/datalab/nfs/deepfm/data/train_21/recommendations/07_08_2025_test_kts_for_dts.pkl.gz"

with gzip.open(recommendations_path, "rb") as f:
    dt_recommendations = pickle.load(f)

In [72]:
recs_nikita = pd.read_excel("/home/datalab/nfs/deepfm/data/test_22_07_2025/bus_recsDT_4testing_22_07.xlsx")

### 100 рекомендаций каждому

In [ ]:
dt_recommendations.keys()

In [80]:
new_rows = []
for dt_index, kt_indices in dt_recommendations.items():
    
    if isinstance(dt_index, str):
        inn_dt = dt_index
    else:
        inn_dt = index_to_inn_dt[dt_index]
    for kt_index in kt_indices:
        if kt_index in index_to_inn_kt:
            inn_kt = index_to_inn_kt[kt_index]
            new_rows.append({"nn_DT": inn_dt, "nn_KT": inn_kt})
            
            
recs_df = pd.DataFrame(new_rows).drop_duplicates().sort_values("nn_DT").reset_index(drop=True)
recs_df["nn_DT"] = recs_df["nn_DT"].astype(str)
# dt_to_process["nn_DT"] = dt_to_process["nn_DT"].astype(str)

In [ ]:
recs_df

In [81]:
result_df = pd.merge(recs_df, dt_to_process, on="nn_DT", how="inner").drop_duplicates().sort_values("nn_DT").reset_index(drop=True)
final_df = pd.concat([dt_to_process, result_df], ignore_index=True).sort_values("nn_DT").reset_index(drop=True)

NameError: name 'dt_to_process' is not defined

In [85]:
recs_df.to_excel("../data/test_07_08_2025/andrey_recs_dt_4testing_07_08.xlsx", index=False)